In [1]:
from tqdm import tqdm
import xarray as xr
import pandas as pd
import warnings
import intake
import gcsfs
import munch
import toml
import sys
import os 

os.chdir('../..')   # beginning of the repository

%load_ext autoreload
%autoreload 2

fs = gcsfs.GCSFileSystem()

In [2]:
config = 'notebooks/gathering/config/pangeo_cmip6_monthly_config.toml'
cmip6_config = munch.munchify(toml.load(config))

col = intake.open_esm_datastore(cmip6_config.pangeo.request.url)

variable_id = cmip6_config.pangeo.request.variable_id
table_id = cmip6_config.pangeo.request.table_id
experiment_id = cmip6_config.pangeo.request.experiment_id
member_id = cmip6_config.pangeo.request.member_id
require_all_on = cmip6_config.pangeo.request.require_all_on
temporal_ranges = cmip6_config.pangeo.ranges
grid_label = cmip6_config.pangeo.request.grid_label

cat = col.search(
    variable_id = variable_id,
    table_id = table_id,
    experiment_id = experiment_id,
    member_id = member_id,
    #grid_label = grid_label,       # most ESMs don't have grid_label defined
    require_all_on = require_all_on,
    )

In [3]:
warnings.filterwarnings("ignore")

dst_path_base = 'data/CMIP6_monthly_data'
if not os.path.exists(dst_path_base): os.makedirs(dst_path_base)
request_length = len(member_id) * len(experiment_id) * len(variable_id)

# Initialize an empty log DataFrame
log_columns = ['ESM', 'Experiment ID', 'Variable ID', 'Member ID', 'Lat', 'Lon', 'Grid Points']
log_df = pd.DataFrame(columns=log_columns)

ESM_df = cat.df
ESMs = ESM_df['source_id'].unique()

# Outer progress bar for Earth System Models
with tqdm(total=len(ESMs), desc="Earth System Models", colour="green", ncols=100, position=0, leave=True) as outer_pbar:
    for ESM_index, ESM in enumerate(ESMs):
        outer_pbar.set_description(f"Processing ESM: {ESM} ({ESM_index + 1}/{len(ESMs)})")
        outer_pbar.update(1)

        requests = ESM_df[
            (ESM_df['source_id'] == ESM) &
            (ESM_df['experiment_id'].isin(experiment_id)) &
            (ESM_df['member_id'].isin(member_id)) &
            (ESM_df['variable_id'].isin(variable_id))]
            #(ESM_df['grid_label'].isin(grid_label))]

        try:
            # Check if the number of records equals the expected number
            #assert request_length == requests.shape[0]
            #if requests.shape[0] >= 1:
            #    requests = requests[requests['grid_label'].isin(grid_label)]

            # Inner progress bar for requests
            with tqdm(total=requests.shape[0], desc=f"Requests for {ESM}", colour="blue", ncols=100, position=1, leave=True) as inner_pbar:
                for request_index, (_, request) in enumerate(requests.iterrows()):
                    dirname = f"{dst_path_base}/{request['member_id']}/{request['variable_id']}/{request['experiment_id']}"
                    if not os.path.exists(dirname): os.makedirs(dirname)
                    fname = f"{request['source_id']}_{request['experiment_id']}_{request['member_id']}_{request['variable_id']}.nc"

                    flag = False
                    if flag:    #os.path.exists(os.path.join(dirname, fname)):
                        pass
                    else:
                        # open the dataset
                        mapper = fs.get_mapper(request['zstore'])
                        dataset = xr.open_zarr(mapper)
                        
                        try:
                            # try to slice temporally
                            ranges = temporal_ranges[request['experiment_id']]
                            dataset = dataset.sel(time=slice(ranges[0], ranges[1]))
                            
                            # Save the file in NetCDF format
                            #dataset.to_netcdf(os.path.join(dirname, fname))

                            if 'latitude' in list(dict(dataset.coords).keys()) and 'longitude' in list(dict(dataset.coords).keys()):
                                latitude = dataset.latitude.size
                                longitude = dataset.longitude.size
                            elif 'lat' in list(dict(dataset.coords).keys()) and 'lon' in list(dict(dataset.coords).keys()):
                                latitude = dataset.lat.size
                                longitude = dataset.lon.size
                            else:
                                ValueError('Latitude and Longitude coords not recognized...')
                                continue

                            # Log relevant details about this request
                            log_df = pd.concat([log_df, pd.DataFrame({
                                'ESM': request['source_id'],
                                'Experiment ID': request['experiment_id'],
                                'Variable ID': request['variable_id'],
                                'Member ID': request['member_id'],
                                'Lat': [int(latitude)],
                                'Lon': [int(longitude)],
                                'Grid Points': [latitude * longitude],
                            })], ignore_index=True)

                        #except AssertionError:
                        except:
                            # TODO fix print and remove files
                            print(f'[!] Issue on model : {request['source_id']} {request['variable_id']}, skip ...')
                            continue

                    # Update inner progress bar
                    inner_pbar.set_description(f"File: {request['member_id']}_{request['source_id']}_{request['experiment_id']}_{request['variable_id']} ({request_index + 1}/{requests.shape[0]})")
                    inner_pbar.update(1)

        except AssertionError:
            # Handle cases where the request length does not match
            outer_pbar.write(f"[!] Mismatch in records for ESM: {ESM}, skip ...")
            continue

# Save the log DataFrame to a CSV file
dirname = 'data/logs'
if not os.path.exists(dirname): os.makedirs(dirname)
log_file_path = os.path.join(dirname, f"esm_log_{variable_id[0]}.csv")
log_df.to_csv(log_file_path, index=False)
print(f"Log saved to {log_file_path}")

File: r1i1p1f1_AWI-CM-1-1-MR_historical_zg (1/1): 100%|███████████████| 1/1 [00:00<00:00,  1.19it/s]
File: r1i1p1f1_AWI-ESM-1-1-LR_historical_zg (1/1): 100%|██████████████| 1/1 [00:00<00:00,  1.38it/s]
File: r1i1p1f1_CESM2-WACCM_historical_zg (1/1): 100%|█████████████████| 1/1 [00:00<00:00,  1.17it/s]
File: r1i1p1f1_CESM2-WACCM-FV2_historical_zg (1/1): 100%|█████████████| 1/1 [00:00<00:00,  1.29it/s]
File: r1i1p1f1_EC-Earth3_historical_zg (1/1): 100%|███████████████████| 1/1 [00:00<00:00,  1.12it/s]
File: r1i1p1f1_EC-Earth3-AerChem_historical_zg (1/1): 100%|███████████| 1/1 [00:00<00:00,  1.35it/s]
File: r1i1p1f1_EC-Earth3-CC_historical_zg (1/1): 100%|████████████████| 1/1 [00:00<00:00,  1.28it/s]
File: r1i1p1f1_EC-Earth3-Veg-LR_historical_zg (1/1): 100%|████████████| 1/1 [00:00<00:00,  1.42it/s]
File: r1i1p1f1_GISS-E2-1-G_historical_zg (1/1): 100%|█████████████████| 1/1 [00:00<00:00,  1.12it/s]
File: r1i1p1f1_GISS-E2-1-G-CC_historical_zg (1/1): 100%|██████████████| 1/1 [00:00<00:00,  

[!] Issue on model : KACE-1-0-G zg, skip ...


File: r1i1p1f1_MIROC6_historical_zg (1/1): 100%|██████████████████████| 1/1 [00:00<00:00,  1.40it/s]
File: r1i1p1f1_MPI-ESM-1-2-HAM_historical_zg (1/1): 100%|█████████████| 1/1 [00:00<00:00,  1.19it/s]
Processing ESM: TaiESM1 (50/50): 100%|██████████████████████████████| 50/50 [00:40<00:00,  1.24it/s]

Log saved to data/logs/esm_log_zg.csv
